In [1]:
spx_impls = ["sha2-128s", "shake-128s",
             "sha2-128f", "shake-128f",
             "sha2-192s", "shake-192s",
             "sha2-192f", "shake-192f",
             "sha2-256s", "shake-256s",
             "sha2-256f", "shake-256f"]

In [2]:
import math

class SPX_PARAMS:
    def __init__(self, PARAMSET, HASH, SECCAT, SPX_N, SPX_LG_W, SPX_H, SPX_D, SPX_A, SPX_K):
        self.PARAMSET = PARAMSET
        self.HASH     = HASH
        self.SECCAT   = SECCAT
        self.SPX_N    = SPX_N
        self.SPX_LG_W = SPX_LG_W
        self.SPX_H    = SPX_H
        self.SPX_D    = SPX_D
        self.SPX_A    = SPX_A
        self.SPX_K    = SPX_K
        
        self.compute_params()
    
    def compute_params(self):
        self.SPX_W = 2**self.SPX_LG_W
        self.SPX_LEN_1 = math.ceil(8*self.SPX_N / self.SPX_LG_W)
        self.SPX_LEN_2 = math.floor(math.log2(self.SPX_LEN_1 * (self.SPX_W - 1)) / self.SPX_LG_W) + 1
        self.SPX_LEN = self.SPX_LEN_1 + self.SPX_LEN_2
        self.SPX_H_PRIME = self.SPX_H // self.SPX_D
        self.SPX_T = 2**self.SPX_A
        self.SPX_M = math.ceil((self.SPX_H - self.SPX_H_PRIME) / 8) + math.ceil(self.SPX_H_PRIME / 8) + math.ceil(self.SPX_K * self.SPX_A / 8)

    def to_file(self):
        with open("params-spx-{paramset}.jinc".format(paramset = self.PARAMSET), "w") as f:
            f.write("")
        with open("params-spx-{paramset}.jinc".format(paramset = self.PARAMSET), "a") as f:
            hash = 0;
            if self.HASH == "sha2":
                hash = 0
            elif self.HASH == "shake":
                hash = 1
            else:
                raise Exception("unknown hash function family: " + self.HASH)
                
            f.write("// hash function family identifiers\n")
            f.write("param int SPX_HASH_SHA2  = 0;\n")
            f.write("param int SPX_HASH_SHAKE = 1;\n")
            f.write("// hash function family used\n")
            f.write("param int SPX_HASHFAM    = " + str(hash) + ";\n")
            f.write("\n")
            
            f.write("// security category of parameter set\n")
            f.write("param int SPX_SEC_CAT = " + str(self.SECCAT) + "; // from SLH-DSS Table 2\n")
            f.write("\n")
            
            f.write("// SPHINCS+ parameters\n")
            f.write("param int SPX_N = " + str(self.SPX_N) + ";\n")
            f.write("param int SPX_M = " + str(self.SPX_M) + "; // ceil((SPX_H - SPX_H_PRIME) / 8) + ceil((SPX_H_PRIME) / 8) + ceil((SPX_K * SPX_A) / 8)\n")
            f.write("param int SPX_SKBYTES = " + str(4*self.SPX_N) + "; // 4 * SPX_N\n")
            f.write("param int SPX_PKBYTES = " + str(2*self.SPX_N) + "; // 2 * SPX_N\n")
            f.write("\n")
            
            f.write("// WOTS+ parameters\n")
            f.write("param int SPX_LG_W  = " + str(self.SPX_LG_W) + ";\n")
            f.write("param int SPX_W     = " + str(self.SPX_W) + "; // 2^SPX_LG_W\n")
            f.write("param int SPX_LEN_1 = " + str(self.SPX_LEN_1) + "; // ceil(8*SPX_N / SPX_LG_W)\n")
            f.write("param int SPX_LEN_2 = " + str(self.SPX_LEN_2) + "; // floor(log2(SPX_LEN_1 * (SPX_W - 1)) / SPX_LG_W) + 1\n")
            f.write("param int SPX_LEN   = " + str(self.SPX_LEN) + "; // SPX_LEN_1 + SPX_LEN_2\n")
            f.write("\n")
            
            f.write("// hypertree and XMSS parameters\n")
            f.write("param int SPX_H       = " + str(self.SPX_H) + ";\n")
            f.write("param int SPX_D       = " + str(self.SPX_D) + ";\n")
            f.write("param int SPX_H_PRIME = " + str(self.SPX_H_PRIME) + "; // SPX_H / SPX_D\n")
            f.write("param int XMSS_TREE_SIZE = " + str(2**(self.SPX_H_PRIME+1)-1) + "; // 2^(SPX_H_PRIME+1)-1\n")
            f.write("\n")
            
            f.write("// FORS parameters\n")
            f.write("param int SPX_A = " + str(self.SPX_A) + ";\n")
            f.write("param int SPX_K = " + str(self.SPX_K) + ";\n")
            f.write("param int SPX_T = " + str(self.SPX_T) + "; // 2^SPX_A\n")
            f.write("param int FORS_TREE_SIZE = " + str(2**(self.SPX_A+1)-1) + "; // 2^(SPX_A+1)-1\n")
            f.write("\n")
            
            f.write("// Signature lengths\n")
            f.write("// WOTS_SIG = one SPX_N-byte signature value for each of the SPX_LEN chains\n")
            f.write("param int WOTS_SIG = " + str(self.SPX_LEN * self.SPX_N) + "; // SPX_LEN * SPX_N\n")
            f.write("// XMSS_SIG = one WOTS+ signature followed by an SPX_H_PRIME-length authentication path\n")
            f.write("param int XMSS_SIG = " + str(self.SPX_LEN*self.SPX_N + self.SPX_H_PRIME*self.SPX_N) + "; // WOTS_SIG + SPX_H_PRIME*SPX_N\n")
            f.write("// HT_SIG = one XMSS signature for each of the SPX_D layers in the hypertree\n")
            f.write("param int HT_SIG   = " + str(self.SPX_D*(self.SPX_LEN*self.SPX_N + self.SPX_H_PRIME*self.SPX_N)) + "; // SPX_D * XMSS_SIG\n")
            f.write("// FORS_SIG = for each of the SPX_K trees in a FORS forest, one SPX_N-byte secret value followed by an SPX_A-length authentication path\n")
            f.write("param int FORS_SIG = " + str(self.SPX_K * (1 + self.SPX_A) * self.SPX_N) + "; // SPX_K * (1 + SPX_A) * SPX_N\n")
            f.write("// SPX_SIG = an SPX_N-byte message randomiser follwed by one FORS signature and one hypertree signature\n")
            f.write("param int SPX_SIG  = " + str(self.SPX_N + self.SPX_D*(self.SPX_LEN*self.SPX_N + self.SPX_H_PRIME*self.SPX_N) + self.SPX_K * (1 + self.SPX_A) * self.SPX_N) + "; // SPX_N + FORS_SIG + HT_SIG\n")
            f.write("\n")

            if self.HASH == "sha2":
                f.write("// SHA2 fixed hash function input and output sizes\n")
                f.write("param int SHA256_IN  = " + str(64) + ";\n")
                f.write("param int SHA256_OUT = " + str(32) + ";\n")
                f.write("param int SHA512_IN  = " + str(128) + ";\n")
                f.write("param int SHA512_OUT = " + str(64) + ";\n")
                f.write("// hash function SHA-X for H_msg, PRF_msg, H, T_len, and T_k, dependent on security category SECCAT\n")
                f.write("// X = 256 if SPX_SEC_CAT == 1, X = 512 if SPX_SEC_CAT == 3 or SPX_SEC_CAT == 5\n")
                f.write("param int SHAX_IN    = " + (str(64) if self.SECCAT == 1 else str(128)) + ";\n")
                f.write("param int SHAX_OUT   = " + (str(32) if self.SECCAT == 1 else str(64)) + ";\n")
                f.write("// hash function SHA-X for H_msg, PRF_msg, H, T_len, and T_k, dependent on security category SECCAT\n")
                f.write("// X = 256 if SPX_SEC_CAT == 1, X = 512 if SPX_SEC_CAT == 3 or SPX_SEC_CAT == 5\n")
                f.write("param int SHA_HMAC_PREFIX = " + (str(64+self.SPX_N) if self.SECCAT == 1 else str(128+self.SPX_N)) + "; // SHAX_IN + SPX_N\n")
                f.write("param int SHA_MGF1_PREFIX = " + str(3*self.SPX_N) + "; // 3*SPX_N\n")
                f.write("\n")
            
            f.write("// word offsets within the ADRS data structure\n")
            f.write("param int SPX_ADRS_OFFSET_LAYERADDR   = 0;\n")
            f.write("param int SPX_ADRS_OFFSET_TREEADDR    = 1;\n")
            f.write("param int SPX_ADRS_OFFSET_TYPE        = 4;\n")
            f.write("param int SPX_ADRS_OFFSET_KEYPAIRADDR = 5;\n")
            f.write("param int SPX_ADRS_OFFSET_CHAINADDR   = 6;\n")
            f.write("param int SPX_ADRS_OFFSET_HASHADDR    = 7;\n")
            f.write("param int SPX_ADRS_OFFSET_TREEHEIGHT  = 6;\n")
            f.write("param int SPX_ADRS_OFFSET_TREEINDEX   = 7;\n")
            f.write("\n")
            
            f.write("// ADRS constant values for the type field\n")
            f.write("param int SPX_ADRS_TYPE_WOTS_HASH  = 0;\n")
            f.write("param int SPX_ADRS_TYPE_WOTS_PK    = 1;\n")
            f.write("param int SPX_ADRS_TYPE_TREE       = 2;\n")
            f.write("param int SPX_ADRS_TYPE_FORS_TREE  = 3;\n")
            f.write("param int SPX_ADRS_TYPE_FORS_ROOTS = 4;\n")
            f.write("param int SPX_ADRS_TYPE_WOTS_PRF   = 5;\n")
            f.write("param int SPX_ADRS_TYPE_FORS_PRF   = 6;")

In [3]:
for paramset in spx_impls:
    spx_params = None
    
    match paramset:
        case "sha2-128s":
            spx_params = SPX_PARAMS(paramset,"sha2",1,16,4,63,7,12,14)
        case "shake-128s":
            spx_params = SPX_PARAMS(paramset,"shake",1,16,4,63,7,12,14)
        case "sha2-128f":
            spx_params = SPX_PARAMS(paramset,"sha2",1,16,4,66,22,6,33)
        case "shake-128f":
            spx_params = SPX_PARAMS(paramset,"shake",1,16,4,66,22,6,33)
        case "sha2-192s":
            spx_params = SPX_PARAMS(paramset,"sha2",3,24,4,63,7,14,17)
        case "shake-192s":
            spx_params = SPX_PARAMS(paramset,"shake",3,24,4,63,7,14,17)
        case "sha2-192f":
            spx_params = SPX_PARAMS(paramset,"sha2",3,24,4,66,22,8,33)
        case "shake-192f":
            spx_params = SPX_PARAMS(paramset,"shake",3,24,4,66,22,8,33)
        case "sha2-256s":
            spx_params = SPX_PARAMS(paramset,"sha2",5,32,4,64,8,14,22)
        case "shake-256s":
            spx_params = SPX_PARAMS(paramset,"shake",5,32,4,64,8,14,22)
        case "sha2-256f":
            spx_params = SPX_PARAMS(paramset,"sha2",5,32,4,68,17,9,35)
        case "shake-256f":
            spx_params = SPX_PARAMS(paramset,"shake",5,32,4,68,17,9,35)
        case _:
            raise Exception("SPHINCS parameter set not recognised: " + str(paramset))
            
    spx_params.to_file()